Mapeamento Regional e Temporal: Identificar quais regiões e estados do Brasil apresentam as maiores taxas de crimes contra mulheres e a população LGBTQIA+, bem como analisar se há uma tendência de aumento ou redução desses índices ao longo dos últimos anos.


Avaliação de Políticas Públicas: Avaliar, através de dados, se a implementação de leis de proteção específicas resultou em impactos reais na diminuição das taxas de criminalidade nas regiões analisadas.

Análise de Tipificação: Investigar a qualidade dos dados oficiais, buscando indícios de falhas de tipificação (como registrar crimes de ódio como crimes comuns).

Implicação Prática: Fornecer um panorama analítico que possa auxiliar gestores de segurança e organizações da sociedade civil na alocação de recursos (como delegacias especializadas) e no aprimoramento dos sistemas de registro de boletins de ocorrência.

# Datas:

**Início da criminalização de crimes contra LGBTQ**:

Desde junho de 2019, o STF, na ADO nº 26, criminalizou a LGBTfobia no Brasil, enquadrando atos de ódio e discriminação contra orientação sexual ou identidade de gênero na Lei de Racismo (Lei 7.716/1989). A conduta é crime inafiançável e imprescritível, com penas de 1 a 3 anos de prisão, podendo ser maior em casos de injúria racial. 
Homicídio Qualificado/Hediondo: A discriminação que motiva um homicídio (LGBTIcídio) torna o crime hediondo, com pena maior (12 a 30 anos).

**Lei do Feminicídio**:

A Lei do Feminicídio (Lei nº 13.104/2015), atualizada pela Lei nº 14.994/2024, torna o assassinato de mulheres por razões de gênero um crime autônomo e hediondo, com penas severas de 20 a 40 anos de reclusão. Considera-se feminicídio quando envolve violência doméstica/familiar ou menosprezo/discriminação à condição de mulher, com agravantes para crimes cometidos na gravidez, contra menores de 14 ou maiores de 60 anos, ou na presença de familiares.

# Perguntas

Quais regiões e estados do Brasil concentram as maiores taxas de violência contra mulheres e pessoas LGBTQIA+, e existe padrão de sazonalidade ou escalonada ao longo dos anos?

Há uma correlação observável entre a implementação de marcos legais de proteção (como a Lei do Feminicídio ou a criminalização da homofobia pelo STF) e a diminuição das ocorrências em estados específicos?

A proporção de feminicídios em relação ao total de homicídios de mulheres indica uma subnotificação ou falha na tipificação do crime em determinadas bases de dados estaduais?

Como a ausência de campos específicos para orientação sexual e identidade de gênero nos registros oficiais impacta a dimensão dos dados quando comparados aos levantamentos de ONGs e observatórios independentes?

In [1]:
from pathlib import Path
from dotenv import dotenv_values

import requests

import pandas as pd
import plotly.express as px

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema
from sqlalchemy import Integer, MetaData, Table, func, select


In [18]:
def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
    return create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=user,
            password=password,
            host=host,
            port=port,
            database=database,
        )
    )


In [19]:
env = dotenv_values(project_root / ".env")
try:
    host = "localhost"
    port = int(env.get("POSTGRES_PORT", 5432))# type:ignore
    database = env["POSTGRES_DB"]
    user = env["POSTGRES_USER"]
    password = env["POSTGRES_PASSWORD"]
    schema_name = "public"
    table_name = "pubseg"
except:
    raise

# SQL

In [20]:
engine = build_postgres_engine(host, port, database, user, password)
metadata = MetaData(schema=schema_name)
pubseg = Table(table_name, metadata, autoload_with=engine, schema=schema_name)

### Tabela por quantidade de ocorrências ao ano

In [21]:
ano = func.extract("year", pubseg.c.data_referencia).cast(Integer)
evento = func.trim(pubseg.c.evento)

evento_counts_stmt = (
    select(
        evento.label("evento"),
        ano.label("ano"),
        func.count().label("quantidade"),
    )
    .where(pubseg.c.evento.is_not(None), evento != "")
    .group_by(evento, ano)
    .order_by(evento, ano)
)

evento_counts_por_ano = pd.read_sql_query(evento_counts_stmt, engine)
evento_counts_por_ano = (
    evento_counts_por_ano
    .pivot(index="evento", columns="ano", values="quantidade")
    .fillna(0)
    .astype(int)
    .reset_index()
)
evento_counts_por_ano.columns.name = None
evento_counts_por_ano

,evento,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Apreensão de Cocaína,636,648,648,648,648,648,648,648,648,648,648
1,Apreensão de Maconha,636,648,648,648,648,648,648,648,648,648,648
2,Arma de Fogo Apreendida,5724,5832,5832,5832,5832,5832,5832,5832,5832,5832,5832
3,Atendimento pré-hospitalar,324,324,324,324,324,324,324,324,324,324,324
4,Busca e salvamento,324,324,324,324,324,324,324,324,324,324,324
5,Combate a incêndios,324,324,324,324,324,324,324,324,324,324,324
6,Emissão de Alvarás de licença,324,324,324,324,324,324,324,324,324,324,324
7,Estupro,324,324,324,324,324,324,324,324,324,324,324
8,Estupro de vulnerável,324,324,324,324,324,324,324,324,324,324,324
9,Feminicídio,66600,67548,67548,67548,67560,67560,67560,67548,67548,67560,67558


plot feito pelo codex

In [22]:
eventos_para_plotar = [
    "Estupro",                                                              
    "Estupro de vulnerável",
    "Feminicídio",
    "Homicídio doloso",
    "Lesão corporal seguida de morte",
    "Tentativa de feminicídio",
    "Tentativa de homicídio",
]

eventos_por_ano_stmt = (
    select(
        ano.label("ano"),
        evento.label("evento"),
        func.count().label("quantidade"),
    )
    .where(evento.in_(eventos_para_plotar))
    .group_by(ano, evento)
    .order_by(ano, evento)
)

eventos_por_ano_df = pd.read_sql_query(eventos_por_ano_stmt, engine)

fig = px.bar(
    eventos_por_ano_df,
    x="ano",
    y="quantidade",
    color="evento",
    barmode="group",
    title="Ocorrências por ano para eventos selecionados",
    labels={"ano": "Ano", "quantidade": "Ocorrências", "evento": "Evento"},
)
fig.show()


# Sem SQL

In [23]:
anos_pubseg = list(range(2015, 2026))
pubseg_download_path = project_root / "data" / "xlsx_dowloads"

pubseg_frames = []
for ano in anos_pubseg:
    caminho_arquivo = pubseg_download_path / f"pubseg-{ano}.xlsx"
    db_ano = pd.read_excel(caminho_arquivo, usecols=["evento", "data_referencia"])
    db_ano["evento"] = db_ano["evento"].astype("string").str.strip()
    db_ano["ano"] = pd.to_datetime(db_ano["data_referencia"], errors="coerce").dt.year
    pubseg_frames.append(db_ano[["evento", "ano"]])

pubseg_xlsx_eventos_df = pd.concat(pubseg_frames, ignore_index=True)
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.dropna(subset=["evento", "ano"])
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.loc[pubseg_xlsx_eventos_df["evento"] != ""]

total_aparicoes_por_evento_ano_xlsx = (
    pubseg_xlsx_eventos_df
    .groupby(["evento", "ano"], as_index=False)
    .size()
    .rename(columns={"size": "total_aparicoes"})
)

total_aparicoes_por_evento_ano_xlsx_tabela = (
    total_aparicoes_por_evento_ano_xlsx
    .pivot(index="evento", columns="ano", values="total_aparicoes")
    .fillna(0)
    .astype(int)
    .reset_index()
)
total_aparicoes_por_evento_ano_xlsx_tabela.columns.name = None
total_aparicoes_por_evento_ano_xlsx_tabela

,evento,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Apreensão de Cocaína,636,648,648,648,648,648,648,648,648,648,648
1,Apreensão de Maconha,636,648,648,648,648,648,648,648,648,648,648
2,Arma de Fogo Apreendida,5724,5832,5832,5832,5832,5832,5832,5832,5832,5832,5832
3,Atendimento pré-hospitalar,324,324,324,324,324,324,324,324,324,324,324
4,Busca e salvamento,324,324,324,324,324,324,324,324,324,324,324
5,Combate a incêndios,324,324,324,324,324,324,324,324,324,324,324
6,Emissão de Alvarás de licença,324,324,324,324,324,324,324,324,324,324,324
7,Estupro,324,324,324,324,324,324,324,324,324,324,324
8,Estupro de vulnerável,324,324,324,324,324,324,324,324,324,324,324
9,Feminicídio,66600,67548,67548,67548,67560,67560,67560,67548,67548,67560,67558


In [ ]:
def download_xlsx(url: str, year: int, download_dir: str | Path) -> Path:
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    download_url = url.format(y=year)
    file_path = download_dir / f"pubseg-{year}.xlsx"
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    file_path.write_bytes(response.content)

    return file_path

In [26]:
from collections import Counter

anos_pubseg = list(range(2015, 2026))
pubseg_download_path = project_root / "data" / "xlsx_dowloads"

pubseg_bruto_por_ano = {
    ano: pd.read_excel(pubseg_download_path / f"pubseg-{ano}.xlsx")
    for ano in anos_pubseg
}

colunas_para_ignorar = {"id", "id_tabela"}
colunas_comparacao = sorted(
    set().union(*(set(df.columns) for df in pubseg_bruto_por_ano.values())) - colunas_para_ignorar
)

def normalizar_pubseg_para_comparacao(df: pd.DataFrame) -> pd.DataFrame:
    db = df.reindex(columns=colunas_comparacao).copy()

    if "data_referencia" in db.columns:
        data_referencia = pd.to_datetime(db["data_referencia"], errors="coerce")
        db["data_referencia"] = data_referencia.dt.strftime("%m-%d")

    for coluna in db.columns:
        serie = db[coluna]
        if pd.api.types.is_datetime64_any_dtype(serie):
            db[coluna] = pd.to_datetime(serie, errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
        elif pd.api.types.is_object_dtype(serie) or pd.api.types.is_string_dtype(serie):
            db[coluna] = serie.astype("string").str.strip()
        else:
            db[coluna] = serie.astype("string")

    return db.fillna("<NA>")

contadores_por_ano = {}
totais_por_ano = {}
for ano, db in pubseg_bruto_por_ano.items():
    db_normalizado = normalizar_pubseg_para_comparacao(db)
    linhas_normalizadas = list(db_normalizado.itertuples(index=False, name=None))
    contadores_por_ano[ano] = Counter(linhas_normalizadas)
    totais_por_ano[ano] = len(linhas_normalizadas)

matriz_sobreposicao_pubseg = pd.DataFrame(index=anos_pubseg, columns=anos_pubseg, dtype=float)

for ano_a in anos_pubseg:
    total_linhas_a = totais_por_ano[ano_a]
    contador_a = contadores_por_ano[ano_a]

    for ano_b in anos_pubseg:
        contador_b = contadores_por_ano[ano_b]
        linhas_iguais = sum(
            min(quantidade_a, contador_b.get(linha, 0))
            for linha, quantidade_a in contador_a.items()
        )
        percentual = 100 * linhas_iguais / total_linhas_a if total_linhas_a else 0
        matriz_sobreposicao_pubseg.loc[ano_a, ano_b] = percentual

matriz_sobreposicao_pubseg = matriz_sobreposicao_pubseg.round(2)
matriz_sobreposicao_pubseg



plot  feito pelo codex

In [ ]:
fig = px.imshow(
    matriz_sobreposicao_pubseg,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Ano B",
        "y": "Ano A",
        "color": "% de linhas de A presentes em B",
    },
    title="Sobreposição percentual entre arquivos pubseg-{ano}.xlsx",
)
fig.update_xaxes(side="top")
fig.show()



,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2015,100.00,81.23,67.51,65.81,61.75,60.63,58.93,58.12,57.70,57.32,57.20
2016,78.68,100.00,68.71,67.15,63.81,62.45,60.68,59.90,58.91,58.45,58.31
2017,72.42,76.09,100.00,81.62,77.53,76.03,74.03,73.18,72.66,72.15,72.06
2018,70.64,74.41,81.66,100.00,79.90,78.60,76.38,75.38,74.94,74.42,74.25
2019,66.24,70.66,77.52,79.85,100.00,82.41,80.45,79.30,78.30,77.80,77.71
2020,65.04,69.15,76.02,78.55,82.41,100.00,81.65,80.46,79.41,78.94,78.81
2021,63.21,67.19,74.02,76.33,80.45,81.65,100.00,81.78,80.94,80.35,79.76
2022,62.35,66.34,73.18,75.34,79.31,80.47,81.79,100.00,81.28,80.71,80.34
2023,61.90,65.25,72.66,74.90,78.31,79.42,80.95,81.28,100.00,80.99,80.38
2024,61.48,64.72,72.14,74.37,77.80,78.94,80.35,80.70,80.98,100.00,80.85


In [2]:
csv_download_path = project_root / "data" / "csv_dowloads"
csv_paths = sorted(csv_download_path.glob("*.csv"))

csv_frames = [
    pd.read_csv(csv_path, sep=";", encoding="latin1", low_memory=False)
    for csv_path in csv_paths
]

disk100_unificado_df = pd.concat(csv_frames, ignore_index=True)



In [3]:
def corrigir_latin1_para_utf8(valor):
    if pd.isna(valor) or not isinstance(valor, str):
        return valor
    try:
        return valor.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return valor

colunas_texto_disk100 = disk100_unificado_df.select_dtypes(include=["object", "string"]).columns

disk100_unificado_df = disk100_unificado_df.copy()
for coluna in colunas_texto_disk100:
    disk100_unificado_df[coluna] = disk100_unificado_df[coluna].map(corrigir_latin1_para_utf8)

colunas_corrigidas_disk100 = [
    corrigir_latin1_para_utf8(coluna).replace("ï»¿", "") if isinstance(coluna, str) else coluna
    for coluna in disk100_unificado_df.columns
]

disk100_unificado_df = disk100_unificado_df.copy()
disk100_unificado_df.columns = colunas_corrigidas_disk100

disk100_unificado_df = disk100_unificado_df.copy()
disk100_unificado_df["Data_de_cadastro"] = pd.to_datetime(
    disk100_unificado_df["Data_de_cadastro"],
    errors="coerce"
)


In [4]:

print(disk100_unificado_df.dtypes.to_string())


hash                                                        str
Data_de_cadastro                                  datetime64[us]
Canal_de_atendimento                                         str
Denúncia_emergencial                                         str
Denunciante                                                  str
Cenário_da_violação                                          str
País                                                         str
UF                                                           str
Município                                                    str
Frequência                                                   str
Início_das_violações                                         str
sl_quantidade_vitimas                                      int64
Grupo_vulnerável                                             str
Motivação                                                    str
Relação_vítima_suspeito                                      str
sl_vitima_cadastro        

In [5]:
grupo_vulneravel_counts = (
    disk100_unificado_df["Grupo_vulnerável"]
    .fillna("Não informado")
    .astype("string")
    .str.strip()
    .replace("", "Não informado")
    .value_counts()
    .rename_axis("Grupo_vulnerável")
    .reset_index(name="quantidade")
)
grupo_vulneravel_counts


,Grupo_vulnerável,quantidade
0,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,1819858
1,VIOLÊNCIA CONTRA PESSOA IDOSA,1059465
2,VIOLÊNCIA CONTRA PESSOA COM DEFICIÊNCIA,757193
3,"VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE",419723
4,VIOLÊNCIA CONTRA A MULHER,390835
5,VIOLÊNCIA CONTRA PESSOA EM RESTRIÇÃO DE LIBERDADE,67235
6,VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+,53238
7,VIOLÊNCIA CONTRA PESSOA EM SITUAÇÃO DE RUA,24851
8,nulo,1


In [ ]:
grupo_vulneravel_mes_a_mes = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["Grupo_vulnerável", "mes"])
    .size()
    .unstack(fill_value=0)
    .sort_index(axis=1)
    .reset_index()
)

grupo_vulneravel_mes_a_mes


In [ ]:
grupos_destacados = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

grupo_vulneravel_mes_plot = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .assign(
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados),
            "Outros",
        ),
        mes=lambda df: df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .groupby(["mes", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_grupos = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.bar(
    grupo_vulneravel_mes_plot,
    x="mes",
    y="quantidade",
    color="grupo_plot",
    barmode="group",
    category_orders={"grupo_plot": ordem_grupos},
    title="Ocorrências por mês: mulher, população LGBTQIA+ e outros",
    labels={"mes": "Mês", "quantidade": "Quantidade de ocorrências", "grupo_plot": "Grupo"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

grupo_vulneravel_mes_plot


In [ ]:
genero_vitima_mes_a_mes = (
    disk100_unificado_df.assign(
        Genero_da_vítima=disk100_unificado_df["Genero_da_vítima"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["Genero_da_vítima", "mes"])
    .size()
    .unstack(fill_value=0)
    .sort_index(axis=1)
    .reset_index()
)

genero_vitima_mes_a_mes


In [ ]:
genero_vitima_mes_plot = (
    disk100_unificado_df.assign(
        Genero_da_vítima=disk100_unificado_df["Genero_da_vítima"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        mes=disk100_unificado_df["Data_de_cadastro"].dt.to_period("M").astype("string"),
    )
    .dropna(subset=["Data_de_cadastro"])
    .groupby(["mes", "Genero_da_vítima"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

fig = px.bar(
    genero_vitima_mes_plot,
    x="mes",
    y="quantidade",
    color="Genero_da_vítima",
    barmode="group",
    title="Ocorrências por mês e gênero da vítima",
    labels={"mes": "Mês", "quantidade": "Quantidade de ocorrências", "Genero_da_vítima": "Gênero da vítima"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

genero_vitima_mes_plot


In [ ]:
grupo_vulneravel_por_uf = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        UF=disk100_unificado_df["UF"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .groupby(["UF", "Grupo_vulnerável"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
    .reset_index()
)

grupo_vulneravel_por_uf


In [ ]:
grupos_destacados_uf = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

grupo_vulneravel_por_uf_plot = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        UF=disk100_unificado_df["UF"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .assign(
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados_uf),
            "Outros",
        )
    )
    .groupby(["UF", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_grupos_uf = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.bar(
    grupo_vulneravel_por_uf_plot,
    x="UF",
    y="quantidade",
    color="grupo_plot",
    barmode="group",
    category_orders={"grupo_plot": ordem_grupos_uf},
    title="Ocorrências por UF: mulher, população LGBTQIA+ e outros",
    labels={"UF": "UF", "quantidade": "Quantidade de ocorrências", "grupo_plot": "Grupo"},
)
fig.show()

grupo_vulneravel_por_uf_plot


In [ ]:
grupo_vulneravel_por_municipio = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        Município=disk100_unificado_df["Município"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .groupby(["Município", "Grupo_vulnerável"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
    .reset_index()
)

grupo_vulneravel_por_municipio


In [ ]:
grupos_destacados_municipio = {
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
}

grupo_vulneravel_por_municipio_plot = (
    disk100_unificado_df.assign(
        Grupo_vulnerável=disk100_unificado_df["Grupo_vulnerável"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
        Município=disk100_unificado_df["Município"]
        .fillna("Não informado")
        .astype("string")
        .str.strip()
        .replace("", "Não informado"),
    )
    .assign(
        grupo_plot=lambda df: df["Grupo_vulnerável"].where(
            df["Grupo_vulnerável"].isin(grupos_destacados_municipio),
            "Outros",
        )
    )
    .groupby(["Município", "grupo_plot"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

ordem_grupos_municipio = [
    "VIOLÊNCIA CONTRA A MULHER",
    "VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+",
    "Outros",
]

fig = px.box(
    grupo_vulneravel_por_municipio_plot,
    x="grupo_plot",
    y="quantidade",
    category_orders={"grupo_plot": ordem_grupos_municipio},
    points="outliers",
    title="Distribuição das ocorrências por município: mulher, população LGBTQIA+ e outros",
    labels={"grupo_plot": "Grupo", "quantidade": "Quantidade de ocorrências por município"},
)
fig.show()

grupo_vulneravel_por_municipio_plot


In [ ]:
grupo_vulneravel_total = (
    grupo_vulneravel_por_uf
    .drop(columns=["UF"])
    .sum(axis=0)
    .reset_index()
    .rename(columns={"index": "Grupo_vulnerável", 0: "quantidade"})
)

fig = px.pie(
    grupo_vulneravel_total,
    names="Grupo_vulnerável",
    values="quantidade",
    title="Proporção total de ocorrências por grupo vulnerável",
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

grupo_vulneravel_total


In [ ]:
grupo_vulneravel_total_ordenado = grupo_vulneravel_total.sort_values("quantidade", ascending=True)

fig = px.bar(
    grupo_vulneravel_total_ordenado,
    x="quantidade",
    y="Grupo_vulnerável",
    orientation="h",
    title="Total de ocorrências por grupo vulnerável",
    labels={"quantidade": "Quantidade de ocorrências", "Grupo_vulnerável": "Grupo vulnerável"},
)
fig.show()

grupo_vulneravel_total_ordenado
